# Part 1 — Ethan Nusinovich: Network construction & validation

Parse the 56 input CSVs, verify dose resolution and per-drug coverage, build the global fitness lookup, build the Fitness Landscape Graph, and produce the hierarchical lattice plot plus the global fitness-distribution boxplot.

In [ ]:
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
import statistics

import matplotlib.pyplot as plt
import numpy as np
import networkx as nx


## Locate the data folder

In [ ]:
# Walk up from the kernel's cwd until we find a sibling data/ folder
# containing the expected dataset layout.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "Raw Data and Growth Rate").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Could not find project root containing data/Raw Data and Growth Rate/")

INPUT_DIR = (PROJECT_ROOT / "data" / "Raw Data and Growth Rate" /
             "Interaction Calculation" / "Data" / "Data" / "input")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"INPUT_DIR    = {INPUT_DIR}")
assert INPUT_DIR.is_dir(), "input/ folder not found inside data/"


## Wide-format CSV parser

In [ ]:
# Each input CSV is one giant single-line file with repeating blocks of:
#   [dose-combo label] + [31 sub-combo header strings] + [31 fitness floats]
# We walk the comma-split fields recovering one block at a time.
DOSE_LABEL_RE = re.compile(
    r"^([A-Z]{3})(\d)([A-Z]{3})(\d)([A-Z]{3})(\d)([A-Z]{3})(\d)([A-Z]{3})(\d)$"
)

@dataclass
class DoseBlock:
    label: str
    doses: dict
    fitness: dict


def parse_input_file(path):
    raw = path.read_text().replace("\n", "")
    fields = raw.split(",")
    blocks = []
    focal_drugs = None
    i = 0
    while i < len(fields):
        tok = fields[i].strip()
        if not tok:
            i += 1
            continue
        m = DOSE_LABEL_RE.match(tok)
        if not m:
            i += 1
            continue
        drugs = [m.group(2 * k + 1) for k in range(5)]
        levels = [int(m.group(2 * k + 2)) for k in range(5)]
        if focal_drugs is None:
            focal_drugs = drugs
        header = fields[i + 1: i + 32]
        try:
            values = [float(v) for v in fields[i + 32: i + 63]]
        except ValueError:
            i += 1
            continue
        fit_map = {frozenset(h.split("+")): v for h, v in zip(header, values)}
        blocks.append(DoseBlock(label=tok, doses=dict(zip(drugs, levels)),
                                 fitness=fit_map))
        i += 63
    if focal_drugs is None:
        raise ValueError(f"No dose blocks found in {path}")
    return focal_drugs, blocks


## Data exploration

In [ ]:
files = sorted(INPUT_DIR.glob("*.csv"))
print(f"Found {len(files)} input CSVs")
assert len(files) == 56, "expected 56 = C(8,5) focal cocktails"


In [ ]:
total_bytes = sum(f.stat().st_size for f in files)
print(f"Total bytes on disk: {total_bytes / 1e6:.1f} MB")
print(f"Average file size:   {total_bytes / len(files) / 1e3:.1f} KB")


In [ ]:
for f in files[:5]:
    print(f"  {f.name}")
print(f"... and {len(files) - 5} more")


In [ ]:
# Peek at the raw format: each CSV is a single long line of label/header/value
# blocks, one block per dose configuration.
raw = files[0].read_text()
print(f"File: {files[0].name}")
print(f"Newlines: {raw.count(chr(10))}, total chars: {len(raw):,}")
print(f"First 200 chars: {raw[:200]}")


## Parse all 56 CSVs

In [ ]:
parsed = []
for i, f in enumerate(files):
    focal, blocks = parse_input_file(f)
    parsed.append((f, focal, blocks))
    if (i + 1) % 10 == 0:
        print(f"  parsed {i + 1}/{len(files)}")
print(f"Done. {len(parsed)} files parsed.")


In [ ]:
f, focal, blocks = parsed[0]
b = blocks[0]
print(f"Focal drugs:        {focal}")
print(f"Blocks in file:     {len(blocks)}")
print(f"First block label:  {b.label}")
print(f"First block doses:  {b.doses}")
print(f"Subsets in block:   {len(b.fitness)}")
print(f"Sample subsets:")
for s in list(b.fitness)[:5]:
    print(f"  {sorted(s)}  ->  {b.fitness[s]:.2f}")


## Verify dose resolution

In [ ]:
block_counts = [len(blocks) for _, _, blocks in parsed]
assert all(c == 243 for c in block_counts), "expected 243 = 3^5 dose configs per file"
print(f"All {len(parsed)} files have exactly 243 dose configurations (3^5).")


In [ ]:
subset_counts = [len(b.fitness) for _, _, blocks in parsed for b in blocks]
assert all(c == 31 for c in subset_counts), "expected 31 = 2^5 - 1 subsets per block"
print(f"All {len(subset_counts):,} blocks carry exactly 31 subset fitness values (2^5 - 1).")


## Verify per-drug coverage

In [ ]:
single_drug_fitness = defaultdict(list)
for _, _, blocks in parsed:
    for b in blocks:
        for d, lvl in b.doses.items():
            single_drug_fitness[(d, lvl)].append(b.fitness[frozenset({d})])

print(f"Distinct (drug, dose) pairs: {len(single_drug_fitness)}")
print(f"Observations per pair: {len(next(iter(single_drug_fitness.values()))):,}")


In [ ]:
drugs = sorted({d for d, _ in single_drug_fitness})
print(f"{'drug':<5} {'dose 1':>8} {'dose 2':>8} {'dose 3':>8}")
for d in drugs:
    means = [statistics.mean(single_drug_fitness[(d, lvl)]) for lvl in (1, 2, 3)]
    print(f"{d:<5} {means[0]:>8.2f} {means[1]:>8.2f} {means[2]:>8.2f}")


## Build the global fitness lookup

In [ ]:
bucket = defaultdict(list)
for _, _, blocks in parsed:
    for b in blocks:
        for subset, fit in b.fitness.items():
            dose_vec = tuple(sorted((d, b.doses[d]) for d in subset))
            bucket[(subset, dose_vec)].append(fit)

lookup = defaultdict(dict)
for (subset, dose_vec), vals in bucket.items():
    lookup[subset][dose_vec] = {
        "mean": statistics.mean(vals),
        "sd": statistics.stdev(vals) if len(vals) > 1 else 0.0,
        "n": len(vals),
    }
lookup = {k: dict(v) for k, v in lookup.items()}
print(f"Distinct subsets:   {len(lookup)}")
print(f"Total entries:      {sum(len(v) for v in lookup.values()):,}")


In [ ]:
size_counts = defaultdict(int)
for subset in lookup:
    size_counts[len(subset)] += 1
expected = {1: 8, 2: 28, 3: 56, 4: 70, 5: 56}  # C(8, k)
print(f"{'size':<6} {'count':>7} {'expected':>10}")
for k in sorted(size_counts):
    print(f"{k:<6} {size_counts[k]:>7} {expected.get(k, '-'):>10}")


## Build the Fitness Landscape Graph

In [ ]:
G = nx.DiGraph()
G.add_node("EMPTY", subset=frozenset(), size=0, fitness=100.0, dose=())

for subset, dose_dict in lookup.items():
    for dose_vec, rec in dose_dict.items():
        nid = "+".join(f"{d}{lvl}" for d, lvl in dose_vec) if dose_vec else "EMPTY"
        if nid == "EMPTY":
            continue
        G.add_node(nid, subset=subset, size=len(subset),
                   fitness=rec["mean"], dose=dose_vec,
                   sd=rec["sd"], n=rec["n"])
print(f"Nodes built: {G.number_of_nodes():,}")


In [ ]:
# Edges: a node connects to neighbors that differ by exactly one drug
# (both add-drug and remove-drug directions). The doses of overlapping
# drugs must match between source and target.
by_size = {}
for nid, data in G.nodes(data=True):
    by_size.setdefault(data["size"], []).append((nid, data))

for size in sorted(by_size):
    if size + 1 not in by_size:
        continue
    for src_id, src in by_size[size]:
        for tgt_id, tgt in by_size[size + 1]:
            if not src["subset"].issubset(tgt["subset"]):
                continue
            if src["subset"]:
                src_dose = dict(src["dose"])
                tgt_dose = dict(tgt["dose"])
                if any(src_dose[d] != tgt_dose[d] for d in src["subset"]):
                    continue
            added = next(iter(tgt["subset"] - src["subset"]))
            grad = tgt["fitness"] - src["fitness"]
            G.add_edge(src_id, tgt_id, direction="add", drug=added, gradient=grad)
            G.add_edge(tgt_id, src_id, direction="remove", drug=added, gradient=-grad)
print(f"Edges built: {G.number_of_edges():,}")


In [ ]:
# Verify against subset combinatorics: 1 + 24 + 252 + 1512 + 5670 + 13608
expected_per_size = {0: 1, 1: 24, 2: 252, 3: 1512, 4: 5670, 5: 13608}
expected_total = sum(expected_per_size.values())
print(f"Expected nodes total: {expected_total:,}")
print(f"Got nodes total:      {G.number_of_nodes():,}")
print()
print(f"{'size':<5} {'expected':>10} {'got':>10}")
got_per_size = defaultdict(int)
for _, d in G.nodes(data=True):
    got_per_size[d["size"]] += 1
for k in sorted(expected_per_size):
    print(f"{k:<5} {expected_per_size[k]:>10,} {got_per_size[k]:>10,}")
assert G.number_of_nodes() == expected_total


## Hierarchical lattice plot for one focal cocktail

In [ ]:
# Pick AMP+CPR+ERY+DOX+TMP at peak dose for all five drugs
focal = ["AMP", "CPR", "ERY", "DOX", "TMP"]
focal_set = frozenset(focal)
doses = {d: 3 for d in focal}

slice_nodes = []
for s in lookup:
    if not s.issubset(focal_set):
        continue
    rec = lookup[s].get(tuple(sorted((d, doses[d]) for d in s)))
    if rec is None:
        continue
    slice_nodes.append((s, rec["mean"]))
print(f"Subsets in this slice: {len(slice_nodes)}")


In [ ]:
fit_map = {s: f for s, f in slice_nodes}
peaks = set()
for s, fs in slice_nodes:
    nbrs = [fit_map[t] for t in fit_map if len(s.symmetric_difference(t)) == 1]
    if nbrs and all(ft <= fs + 1e-9 for ft in nbrs):
        peaks.add(s)

by_size_local = {}
for s, _ in slice_nodes:
    by_size_local.setdefault(len(s), []).append(s)
pos = {}
for sz, items in by_size_local.items():
    items_sorted = sorted(items, key=lambda s: tuple(sorted(s)))
    n = len(items_sorted)
    for i, s in enumerate(items_sorted):
        x = (i - (n - 1) / 2) / max(n - 1, 1) if n > 1 else 0.0
        pos[s] = (x, sz)

xs = np.array([pos[s][0] for s, _ in slice_nodes])
ys = np.array([pos[s][1] for s, _ in slice_nodes])
fits = np.array([f for _, f in slice_nodes])

fig, ax = plt.subplots(figsize=(13, 6.5))
for s, _ in slice_nodes:
    for t, _ in slice_nodes:
        if t.issuperset(s) and len(t) - len(s) == 1:
            ax.plot([pos[s][0], pos[t][0]], [pos[s][1], pos[t][1]],
                    color="#cccccc", lw=0.4, alpha=0.6, zorder=1)

edge_widths = [2.5 if s in peaks else 0.4 for s, _ in slice_nodes]
edge_colors = ["black" if s in peaks else "white" for s, _ in slice_nodes]
sc = ax.scatter(xs, ys, c=fits, cmap="viridis", s=350,
                edgecolors=edge_colors, linewidths=edge_widths, zorder=2)
for s, _ in slice_nodes:
    if s in peaks:
        ax.text(pos[s][0], pos[s][1], "+".join(sorted(s)),
                fontsize=7, ha="center", va="center", color="white", zorder=3)
cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Fitness (% of drug-free control)")
ax.set_yticks(sorted(by_size_local))
ax.set_yticklabels([f"{k} drug{'s' if k != 1 else ''}" for k in sorted(by_size_local)])
ax.set_xticks([])
ax.set_xlabel("Subsets at this layer (sorted)")
ax.set_title(f"Lattice for {'+'.join(focal)} at peak dose — "
             f"{len(peaks)} local fitness peak(s) outlined")
plt.tight_layout()
plt.show()


## Global fitness-distribution boxplot

In [ ]:
fit_by_size = defaultdict(list)
for _, data in G.nodes(data=True):
    fit_by_size[data["size"]].append(data["fitness"])

sizes = sorted(fit_by_size)
fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot([fit_by_size[s] for s in sizes], positions=sizes,
                widths=0.6, patch_artist=True, showfliers=False)
for patch in bp["boxes"]:
    patch.set_facecolor("#7da6ff")
ax.axhline(100, color="gray", ls="--", lw=0.8, label="Drug-free control")
ax.set_xticks(sizes)
ax.set_xlabel("Subset size (# drugs in combination)")
ax.set_ylabel("Fitness (% of control)")
ax.set_title("Fitness distribution across the Fitness Landscape Graph")
ax.legend()
plt.tight_layout()
plt.show()
